In [28]:
import os
import glob
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

warnings.filterwarnings("ignore")

In [29]:
DATASET_ROOT = r".\data"
OUTPUT_ROOT = r".\prepared_balanced_nilm"

BUILDINGS = ["building_01", "building_02", "building_03", "building_04"]
SHARED_APPLIANCES = ["fridge", "washing_machine", "dishwasher"]

SEED = 42
CLIP_PERCENTILE = 99.5
EPS = 1e-8

WINDOW_SIZE = 129
HALF = WINDOW_SIZE // 2

USE_MEDIAN_FILTER = False
USE_LOG_TARGET = True

ON_THRESHOLDS = {
    "fridge": 10.0,
    "washing_machine": 20.0,
    "dishwasher": 20.0
}

BASE_STRIDE_INACTIVE = 240
BASE_STRIDE_ACTIVE = 30
BASE_STRIDE_EVENT = 1

MAX_INACTIVE_SAMPLES = 20000
MAX_ACTIVE_SAMPLES = 25000
MAX_EVENT_SAMPLES = 25000

EVENT_CONTEXT_RADIUS = 64

random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)

In [30]:
def read_h5_series(file_path):
    with h5py.File(file_path, "r") as f:
        candidates = []

        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                candidates.append((name, obj.shape, obj.dtype))

        f.visititems(visitor)

        numeric_candidates = []
        for name, shape, dtype in candidates:
            dt = np.dtype(dtype)
            if np.issubdtype(dt, np.number):
                numeric_candidates.append((name, shape, dtype))

        if len(numeric_candidates) == 0:
            raise ValueError(f"No numeric dataset found in {file_path}")

        best_name = None
        best_size = -1
        for name, shape, dtype in numeric_candidates:
            size = int(np.prod(shape)) if len(shape) > 0 else 1
            if size > best_size:
                best_size = size
                best_name = name

        arr = f[best_name][()]

    arr = np.asarray(arr).squeeze()

    if arr.ndim == 0:
        arr = np.array([arr], dtype=np.float32)
    elif arr.ndim > 1:
        if arr.shape[1] > 1:
            arr = arr[:, 0]
        else:
            arr = arr.reshape(-1)

    arr = np.asarray(arr, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return arr


In [31]:
def median_filter_1d(x, k=3):
    if k <= 1:
        return x.copy()
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode="edge")
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = np.median(xp[i:i+k])
    return out

def clip_series(x, percentile=99.5):
    upper = np.percentile(x, percentile)
    return np.clip(x, 0.0, upper).astype(np.float32), float(upper)

def zscore(x):
    mu = float(np.mean(x))
    sigma = float(np.std(x)) + EPS
    z = (x - mu) / sigma
    return z.astype(np.float32), mu, sigma

def log1p_transform(x):
    return np.log1p(np.maximum(x, 0.0)).astype(np.float32)

In [32]:
def list_available_appliances(building_path):
    files = sorted(glob.glob(os.path.join(building_path, "*.h5")))
    return [os.path.basename(f).replace(".h5", "").lower() for f in files]

def load_shared_appliances_for_building(building_name, apply_filter=False):
    building_path = os.path.join(DATASET_ROOT, building_name)
    available = list_available_appliances(building_path)

    missing = [ap for ap in SHARED_APPLIANCES if ap not in available]
    if len(missing) > 0:
        raise ValueError(
            f"{building_name} is missing shared appliances: {missing}. "
            f"Available: {available}"
        )

    series = {}
    for ap in SHARED_APPLIANCES:
        file_path = os.path.join(building_path, f"{ap}.h5")
        arr = read_h5_series(file_path)
        if apply_filter:
            arr = median_filter_1d(arr, k=3)
        series[ap] = arr.astype(np.float32)

    min_len = min(len(v) for v in series.values())
    for ap in series:
        series[ap] = series[ap][:min_len]

    return series

In [33]:
def make_active_mask(y, threshold):
    return (y > threshold).astype(np.uint8)

def make_event_mask(y, threshold):
    active = make_active_mask(y, threshold)
    event = np.zeros_like(active, dtype=np.uint8)
    event[1:] = (active[1:] != active[:-1]).astype(np.uint8)
    return event

def expand_event_region(event_idx, length, radius):
    mask = np.zeros(length, dtype=np.uint8)
    for idx in event_idx:
        start = max(0, idx - radius)
        end = min(length, idx + radius + 1)
        mask[start:end] = 1
    return mask

In [34]:
def valid_centers(length, half):
    return np.arange(half, length - half, dtype=np.int64)

def subsample_indices(indices, stride=None, max_samples=None):
    indices = np.asarray(indices, dtype=np.int64)
    if stride is not None and stride > 1:
        indices = indices[::stride]
    if max_samples is not None and len(indices) > max_samples:
        indices = np.sort(rng.choice(indices, size=max_samples, replace=False))
    return np.sort(indices)

def build_balanced_indices(y_raw, threshold, half):
    n = len(y_raw)
    centers = valid_centers(n, half)

    active_mask = make_active_mask(y_raw, threshold)
    event_mask = make_event_mask(y_raw, threshold)

    event_idx = np.where(event_mask == 1)[0]
    event_region = expand_event_region(event_idx, n, EVENT_CONTEXT_RADIUS)

    active_centers = centers[active_mask[centers] == 1]
    event_centers = centers[event_region[centers] == 1]
    inactive_centers = centers[(active_mask[centers] == 0) & (event_region[centers] == 0)]

    active_centers = subsample_indices(active_centers, stride=BASE_STRIDE_ACTIVE, max_samples=MAX_ACTIVE_SAMPLES)
    event_centers = subsample_indices(event_centers, stride=BASE_STRIDE_EVENT, max_samples=MAX_EVENT_SAMPLES)
    inactive_centers = subsample_indices(inactive_centers, stride=BASE_STRIDE_INACTIVE, max_samples=MAX_INACTIVE_SAMPLES)

    balanced = np.unique(np.concatenate([inactive_centers, active_centers, event_centers]))
    balanced = np.sort(balanced)

    return {
        "all_centers": centers,
        "active_centers": active_centers,
        "event_centers": event_centers,
        "inactive_centers": inactive_centers,
        "balanced_centers": balanced
    }

In [35]:
def make_regression_weights(y_raw, threshold):
    y = np.asarray(y_raw, dtype=np.float32)

    active = (y > threshold).astype(np.float32)

    pos = y[y > threshold]
    if len(pos) > 0:
        q50 = np.percentile(pos, 50)
        q90 = np.percentile(pos, 90)
    else:
        q50 = threshold
        q90 = threshold

    weights = np.ones_like(y, dtype=np.float32)
    weights += 2.0 * active
    weights += 1.0 * (y > q50).astype(np.float32)
    weights += 2.0 * (y > q90).astype(np.float32)

    return weights.astype(np.float32)

In [36]:
def prepare_building(building_name):
    print(f"\nPreparing {building_name} ...")

    series = load_shared_appliances_for_building(
        building_name,
        apply_filter=USE_MEDIAN_FILTER
    )

    raw_df = pd.DataFrame(series)
    raw_df["aggregate"] = raw_df[SHARED_APPLIANCES].sum(axis=1)

    clipped = {}
    clip_caps = {}

    for col in ["aggregate"] + SHARED_APPLIANCES:
        clipped[col], cap = clip_series(raw_df[col].values.astype(np.float32), percentile=CLIP_PERCENTILE)
        clip_caps[col] = cap

    clipped_df = pd.DataFrame(clipped)

    agg_norm, agg_mu, agg_std = zscore(clipped_df["aggregate"].values.astype(np.float32))

    out_df = pd.DataFrame({
        "aggregate_raw": raw_df["aggregate"].values.astype(np.float32),
        "aggregate_clipped": clipped_df["aggregate"].values.astype(np.float32),
        "aggregate_norm": agg_norm
    })

    metadata = {
        "building": building_name,
        "length": int(len(raw_df)),
        "window_size": WINDOW_SIZE,
        "clip_percentile": CLIP_PERCENTILE,
        "shared_appliances": SHARED_APPLIANCES,
        "aggregate_stats": {
            "mean": agg_mu,
            "std": agg_std,
            "clip_cap": clip_caps["aggregate"]
        },
        "targets": {}
    }

    building_out = os.path.join(OUTPUT_ROOT, building_name)
    Path(building_out).mkdir(parents=True, exist_ok=True)

    for ap in SHARED_APPLIANCES:
        y_raw = raw_df[ap].values.astype(np.float32)
        y_clip = clipped_df[ap].values.astype(np.float32)

        y_for_norm = log1p_transform(y_clip) if USE_LOG_TARGET else y_clip
        y_norm, y_mu, y_std = zscore(y_for_norm)

        active_mask = make_active_mask(y_raw, ON_THRESHOLDS[ap])
        event_mask = make_event_mask(y_raw, ON_THRESHOLDS[ap])
        weights = make_regression_weights(y_raw, ON_THRESHOLDS[ap])

        idx_info = build_balanced_indices(y_raw, ON_THRESHOLDS[ap], HALF)

        out_df[f"{ap}_raw"] = y_raw
        out_df[f"{ap}_clipped"] = y_clip
        out_df[f"{ap}_norm"] = y_norm
        out_df[f"{ap}_active"] = active_mask
        out_df[f"{ap}_event"] = event_mask
        out_df[f"{ap}_weight"] = weights

        np.save(os.path.join(building_out, f"{ap}_all_centers.npy"), idx_info["all_centers"])
        np.save(os.path.join(building_out, f"{ap}_active_centers.npy"), idx_info["active_centers"])
        np.save(os.path.join(building_out, f"{ap}_event_centers.npy"), idx_info["event_centers"])
        np.save(os.path.join(building_out, f"{ap}_inactive_centers.npy"), idx_info["inactive_centers"])
        np.save(os.path.join(building_out, f"{ap}_balanced_centers.npy"), idx_info["balanced_centers"])

        metadata["targets"][ap] = {
            "threshold": ON_THRESHOLDS[ap],
            "clip_cap": clip_caps[ap],
            "mean": y_mu,
            "std": y_std,
            "log_target": bool(USE_LOG_TARGET),
            "active_ratio": float(active_mask.mean()),
            "event_ratio": float(event_mask.mean()),
            "n_all_centers": int(len(idx_info["all_centers"])),
            "n_active_centers": int(len(idx_info["active_centers"])),
            "n_event_centers": int(len(idx_info["event_centers"])),
            "n_inactive_centers": int(len(idx_info["inactive_centers"])),
            "n_balanced_centers": int(len(idx_info["balanced_centers"]))
        }

    csv_path = os.path.join(building_out, "prepared_timeseries.csv")
    meta_path = os.path.join(building_out, "metadata.json")

    out_df.to_csv(csv_path, index=False)
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved: {csv_path}")
    print(f"Saved: {meta_path}")

    for ap in SHARED_APPLIANCES:
        info = metadata["targets"][ap]
        print(
            f"{ap:16s} | active={info['active_ratio']:.4f} | "
            f"event={info['event_ratio']:.4f} | "
            f"balanced={info['n_balanced_centers']}"
        )

    return metadata

In [37]:
all_meta = []

for b in BUILDINGS:
    try:
        meta = prepare_building(b)
        all_meta.append(meta)
    except Exception as e:
        print(f"[ERROR] {b}: {e}")

summary_rows = []
for meta in all_meta:
    row = {
        "building": meta["building"],
        "length": meta["length"]
    }
    for ap in SHARED_APPLIANCES:
        t = meta["targets"][ap]
        row[f"{ap}_active_ratio"] = t["active_ratio"]
        row[f"{ap}_event_ratio"] = t["event_ratio"]
        row[f"{ap}_balanced_centers"] = t["n_balanced_centers"]
        row[f"{ap}_active_centers"] = t["n_active_centers"]
        row[f"{ap}_event_centers"] = t["n_event_centers"]
        row[f"{ap}_inactive_centers"] = t["n_inactive_centers"]
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUTPUT_ROOT, "summary.csv")
summary_df.to_csv(summary_path, index=False)

print("\nSaved summary:", summary_path)
print(summary_df)


Preparing building_01 ...
Saved: .\prepared_balanced_nilm\building_01\prepared_timeseries.csv
Saved: .\prepared_balanced_nilm\building_01\metadata.json
fridge           | active=0.4820 | event=0.0026 | balanced=54793
washing_machine  | active=0.0020 | event=0.0001 | balanced=31576
dishwasher       | active=0.0025 | event=0.0000 | balanced=28353

Preparing building_02 ...
Saved: .\prepared_balanced_nilm\building_02\prepared_timeseries.csv
Saved: .\prepared_balanced_nilm\building_02\metadata.json
fridge           | active=0.2536 | event=0.0101 | balanced=57965
washing_machine  | active=0.0348 | event=0.0023 | balanced=55291
dishwasher       | active=0.0298 | event=0.0006 | balanced=53819

Preparing building_03 ...
Saved: .\prepared_balanced_nilm\building_03\prepared_timeseries.csv
Saved: .\prepared_balanced_nilm\building_03\metadata.json
fridge           | active=0.4978 | event=0.0017 | balanced=55987
washing_machine  | active=0.0498 | event=0.0021 | balanced=45374
dishwasher       | ac